In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_utils import load_digits_data
from src.pca import PCAFromScratch
from src.kmeans import KMeansFromScratch
from src.gmm import GMMEMFromScratch
from src.metrics import (
    clustering_accuracy,
    remap_cluster_labels,
    assignment_entropy,
    evaluate_clustering,
    print_metrics_table,
)
from src.plotting import (
    plot_pca_scatter,
    plot_log_likelihood,
    plot_entropy_hist,
    show_digit_grid,
)
from src.diagnostics import covariance_floor_ratio

In [ ]:
def run_single_clustering_experiment(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
    representation_name: str,
    model_name: str,
    random_state: int = 42,
    gmm_reg_covar: float = 1e-4,
    gmm_n_init: int = 20,
):
    """
    Run one experiment for a fixed representation + clustering model.
    Returns a dict suitable for DataFrame aggregation.
    """
    row = {
        "Representation": representation_name,
        "Model": model_name,
    }

    if model_name == "KMeans":
        model = KMeansFromScratch(
            n_clusters=10,
            max_iter=100,
            tol=1e-4,
            n_init=20,
            random_state=random_state,
        )
        train_pred = model.fit_predict(X_train)
        test_pred = model.predict(X_test)

        train_metrics = evaluate_clustering(X_train, y_train, train_pred)
        test_metrics = evaluate_clustering(X_test, y_test, test_pred)

        row.update({
            "Train_ACC": train_metrics["ACC"],
            "Train_ARI": train_metrics["ARI"],
            "Train_NMI": train_metrics["NMI"],
            "Train_Silhouette": train_metrics["Silhouette"],
            "Train_MeanEntropy": np.nan,
            "Train_StdEntropy": np.nan,
            "Train_Objective": model.inertia_,
            "Test_ACC": test_metrics["ACC"],
            "Test_ARI": test_metrics["ARI"],
            "Test_NMI": test_metrics["NMI"],
            "Test_Silhouette": test_metrics["Silhouette"],
            "Test_MeanEntropy": np.nan,
            "Test_StdEntropy": np.nan,
            "Test_Objective": np.nan,
            "Converged": getattr(model.result_, "converged", np.nan),
            "Iterations": getattr(model.result_, "n_iter", np.nan),
            "CovFloorRatio": np.nan,
        })

        extras = {
            "model": model,
            "train_pred": train_pred,
            "test_pred": test_pred,
            "test_resp": None,
        }

    elif model_name == "GMM":
        model = GMMEMFromScratch(
            n_components=10,
            max_iter=200,
            tol=1e-4,
            reg_covar=gmm_reg_covar,
            n_init=gmm_n_init,
            init_method="kmeans",
            random_state=random_state,
        )
        train_pred = model.fit_predict(X_train)
        test_pred = model.predict(X_test)
        test_resp = model.predict_proba(X_test)

        train_metrics = evaluate_clustering(
            X_train, y_train, train_pred, responsibilities=model.responsibilities_
        )
        test_metrics = evaluate_clustering(
            X_test, y_test, test_pred, responsibilities=test_resp
        )

        row.update({
            "Train_ACC": train_metrics["ACC"],
            "Train_ARI": train_metrics["ARI"],
            "Train_NMI": train_metrics["NMI"],
            "Train_Silhouette": train_metrics["Silhouette"],
            "Train_MeanEntropy": train_metrics["MeanEntropy"],
            "Train_StdEntropy": train_metrics["StdEntropy"],
            "Train_Objective": model.log_likelihood_history_[-1],
            "Test_ACC": test_metrics["ACC"],
            "Test_ARI": test_metrics["ARI"],
            "Test_NMI": test_metrics["NMI"],
            "Test_Silhouette": test_metrics["Silhouette"],
            "Test_MeanEntropy": test_metrics["MeanEntropy"],
            "Test_StdEntropy": test_metrics["StdEntropy"],
            "Test_Objective": np.nan,
            "Converged": model.result_.converged,
            "Iterations": model.result_.n_iter,
            "CovFloorRatio": covariance_floor_ratio(model.covariances_, gmm_reg_covar),
        })

        extras = {
            "model": model,
            "train_pred": train_pred,
            "test_pred": test_pred,
            "test_resp": test_resp,
        }

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return row, extras

In [ ]:
import pandas as pd

def run_c1_experiments(
    pca_dims=(2, 10, 20, 30),
    random_state: int = 42,
    gmm_reg_covar: float = 1e-4,
    gmm_n_init: int = 20,
):
    """
    Main loop for C1:
    Raw / PCA-d representations x {KMeans, GMM}
    """
    data = load_digits_data(test_size=0.25, random_state=random_state, standardize=True)

    X_train_raw = data["X_train"]
    X_test_raw = data["X_test"]
    y_train = data["y_train"]
    y_test = data["y_test"]
    images_test = data["images_test"]

    representations = {
        "Raw-64": (X_train_raw, X_test_raw, None)
    }

    for dim in pca_dims:
        pca = PCAFromScratch(n_components=dim)
        X_train_pca = pca.fit_transform(X_train_raw)
        X_test_pca = pca.transform(X_test_raw)
        representations[f"PCA-{dim}"] = (X_train_pca, X_test_pca, pca)

    results = []
    artifacts = {}

    for rep_name, (Xtr, Xte, pca_obj) in representations.items():
        for model_name in ["KMeans", "GMM"]:
            print(f"Running: {rep_name} + {model_name}")

            row, extras = run_single_clustering_experiment(
                X_train=Xtr,
                X_test=Xte,
                y_train=y_train,
                y_test=y_test,
                representation_name=rep_name,
                model_name=model_name,
                random_state=random_state,
                gmm_reg_covar=gmm_reg_covar,
                gmm_n_init=gmm_n_init,
            )

            if pca_obj is not None:
                row["PCA_ExplainedVarRatioSum"] = float(np.sum(pca_obj.explained_variance_ratio_))
                row["PCA_ReconError_Train"] = float(pca_obj.reconstruction_error(X_train_raw))
            else:
                row["PCA_ExplainedVarRatioSum"] = np.nan
                row["PCA_ReconError_Train"] = np.nan

            results.append(row)
            artifacts[(rep_name, model_name)] = {
                "extras": extras,
                "pca": pca_obj,
                "X_train": Xtr,
                "X_test": Xte,
                "y_train": y_train,
                "y_test": y_test,
                "images_test": images_test,
            }

    results_df = pd.DataFrame(results)

    desired_cols = [
        "Representation", "Model",
        "Train_ACC", "Train_ARI", "Train_NMI", "Train_Silhouette",
        "Train_MeanEntropy", "Train_StdEntropy", "Train_Objective",
        "Test_ACC", "Test_ARI", "Test_NMI", "Test_Silhouette",
        "Test_MeanEntropy", "Test_StdEntropy",
        "Converged", "Iterations", "CovFloorRatio",
        "PCA_ExplainedVarRatioSum", "PCA_ReconError_Train"
    ]
    results_df = results_df[desired_cols]

    return results_df, artifacts

In [ ]:
results_df, artifacts = run_c1_experiments(
    pca_dims=(2, 10, 20, 30),
    random_state=42,
    gmm_reg_covar=1e-4,
    gmm_n_init=20,
)

results_df

In [ ]:
results_sorted = results_df.sort_values(
    by=["Test_ARI", "Test_NMI", "Test_ACC"],
    ascending=False
).reset_index(drop=True)

results_sorted

In [ ]:
report_table = results_df[
    [
        "Representation", "Model",
        "Test_ACC", "Test_ARI", "Test_NMI", "Test_Silhouette",
        "Test_MeanEntropy",
        "CovFloorRatio",
        "PCA_ExplainedVarRatioSum",
    ]
].copy()

report_table = report_table.sort_values(
    by=["Representation", "Model"]
).reset_index(drop=True)

report_table

In [ ]:
def _extract_dim(rep_name: str) -> int:
    if rep_name.startswith("Raw"):
        return 64
    return int(rep_name.split("-")[1])


def plot_metric_vs_dimension(
    results_df: pd.DataFrame,
    metric: str,
    title: str,
    include_raw: bool = True,
):
    plt.figure(figsize=(7, 5))

    for model_name in ["KMeans", "GMM"]:
        sub = results_df[results_df["Model"] == model_name].copy()
        sub["Dim"] = sub["Representation"].apply(_extract_dim)
        sub = sub.sort_values("Dim")

        if not include_raw:
            sub = sub[sub["Representation"] != "Raw-64"]

        plt.plot(sub["Dim"], sub[metric], marker="o", label=model_name)

    plt.xlabel("Representation Dimension")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_grouped_bar(
    results_df: pd.DataFrame,
    metric: str,
    title: str,
):
    labels = list(results_df["Representation"].unique())
    x = np.arange(len(labels))
    width = 0.36

    km_vals = []
    gmm_vals = []

    for rep in labels:
        km_vals.append(results_df[
            (results_df["Representation"] == rep) & (results_df["Model"] == "KMeans")
        ][metric].values[0])
        gmm_vals.append(results_df[
            (results_df["Representation"] == rep) & (results_df["Model"] == "GMM")
        ][metric].values[0])

    plt.figure(figsize=(9, 5))
    plt.bar(x - width/2, km_vals, width, label="KMeans")
    plt.bar(x + width/2, gmm_vals, width, label="GMM")
    plt.xticks(x, labels)
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_grouped_bar(results_df, "Test_ACC", "C1 Test ACC: KMeans vs GMM across Representations")
plot_grouped_bar(results_df, "Test_ARI", "C1 Test ARI: KMeans vs GMM across Representations")
plot_grouped_bar(results_df, "Test_NMI", "C1 Test NMI: KMeans vs GMM across Representations")

plot_metric_vs_dimension(results_df, "Test_ARI", "Test ARI vs Representation Dimension")
plot_metric_vs_dimension(results_df, "Test_NMI", "Test NMI vs Representation Dimension")
plot_metric_vs_dimension(results_df, "Test_ACC", "Test ACC vs Representation Dimension")

In [ ]:
gmm_rows = results_df[results_df["Model"] == "GMM"].copy()
gmm_rows["Dim"] = gmm_rows["Representation"].apply(_extract_dim)
gmm_rows = gmm_rows.sort_values("Dim")

plt.figure(figsize=(7, 5))
plt.plot(gmm_rows["Dim"], gmm_rows["Test_MeanEntropy"], marker="o")
plt.xlabel("Representation Dimension")
plt.ylabel("Mean Entropy")
plt.title("GMM Mean Assignment Entropy vs Representation Dimension")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(gmm_rows["Dim"], gmm_rows["CovFloorRatio"], marker="o")
plt.xlabel("Representation Dimension")
plt.ylabel("Covariance Floor Ratio")
plt.title("GMM Covariance Floor Ratio vs Representation Dimension")
plt.tight_layout()
plt.show()

In [ ]:
def visualize_test_clusters_on_pca2(
    artifacts: dict,
    target_representation: str,
    target_model: str,
):
    """
    Show clustering result in the 2D PCA space for the test set.
    Even if the model was trained in higher-dimensional representation,
    we still display points using PCA-2 coordinates from the raw data.
    """
    # Always use the PCA-2 embedding from artifacts for visualization
    pca2_art = artifacts[("PCA-2", "KMeans")]  # representation holder only
    data_ref = load_digits_data(test_size=0.25, random_state=42, standardize=True)
    X_train_raw = data_ref["X_train"]
    X_test_raw = data_ref["X_test"]
    y_test = data_ref["y_test"]

    pca2 = PCAFromScratch(n_components=2)
    pca2.fit(X_train_raw)
    Z_test_2 = pca2.transform(X_test_raw)

    target = artifacts[(target_representation, target_model)]
    test_pred = target["extras"]["test_pred"]

    remapped = remap_cluster_labels(y_test, test_pred)

    plot_pca_scatter(
        Z_test_2,
        remapped,
        title=f"{target_representation} + {target_model} (shown in PCA-2 space)"
    )

In [ ]:
visualize_test_clusters_on_pca2(artifacts, "Raw-64", "KMeans")
visualize_test_clusters_on_pca2(artifacts, "Raw-64", "GMM")
visualize_test_clusters_on_pca2(artifacts, "PCA-10", "KMeans")
visualize_test_clusters_on_pca2(artifacts, "PCA-10", "GMM")

In [ ]:
results_df.to_csv("c1_results_full.csv", index=False)
report_table.to_csv("c1_results_report_table.csv", index=False)

print("Saved:")
print(" - c1_results_full.csv")
print(" - c1_results_report_table.csv")

In [ ]:
import os

FIG_DIR = "../results/figures/c1_representation"
TAB_DIR = "../results/tables"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

In [ ]:
def plot_grouped_bar(results_df, metric, title, save_path=None):
    labels = list(results_df["Representation"].unique())
    x = np.arange(len(labels))
    width = 0.36

    km_vals = []
    gmm_vals = []

    for rep in labels:
        km_vals.append(results_df[
            (results_df["Representation"] == rep) & (results_df["Model"] == "KMeans")
        ][metric].values[0])
        gmm_vals.append(results_df[
            (results_df["Representation"] == rep) & (results_df["Model"] == "GMM")
        ][metric].values[0])

    plt.figure(figsize=(9, 5))
    plt.bar(x - width/2, km_vals, width, label="KMeans")
    plt.bar(x + width/2, gmm_vals, width, label="GMM")
    plt.xticks(x, labels)
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")

    plt.show()

In [ ]:
def plot_metric_vs_dimension(results_df, metric, title, save_path=None):
    plt.figure(figsize=(7, 5))

    for model_name in ["KMeans", "GMM"]:
        sub = results_df[results_df["Model"] == model_name].copy()
        sub["Dim"] = sub["Representation"].apply(_extract_dim)
        sub = sub.sort_values("Dim")

        plt.plot(sub["Dim"], sub[metric], marker="o", label=model_name)

    plt.xlabel("Dimension")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")

    plt.show()

In [ ]:
# ===== 柱状图（报告主图）=====
plot_grouped_bar(
    results_df, "Test_ACC",
    "C1 Test ACC",
    save_path=f"{FIG_DIR}/c1_test_acc.png"
)

plot_grouped_bar(
    results_df, "Test_ARI",
    "C1 Test ARI",
    save_path=f"{FIG_DIR}/c1_test_ari.png"
)

plot_grouped_bar(
    results_df, "Test_NMI",
    "C1 Test NMI",
    save_path=f"{FIG_DIR}/c1_test_nmi.png"
)


# ===== 折线图（趋势分析）=====
plot_metric_vs_dimension(
    results_df, "Test_ARI",
    "ARI vs Dimension",
    save_path=f"{FIG_DIR}/c1_ari_vs_dim.png"
)

plot_metric_vs_dimension(
    results_df, "Test_NMI",
    "NMI vs Dimension",
    save_path=f"{FIG_DIR}/c1_nmi_vs_dim.png"
)

plot_metric_vs_dimension(
    results_df, "Test_ACC",
    "ACC vs Dimension",
    save_path=f"{FIG_DIR}/c1_acc_vs_dim.png"
)

In [ ]:
gmm_rows = results_df[results_df["Model"] == "GMM"].copy()
gmm_rows["Dim"] = gmm_rows["Representation"].apply(_extract_dim)
gmm_rows = gmm_rows.sort_values("Dim")

# entropy
plt.figure(figsize=(7, 5))
plt.plot(gmm_rows["Dim"], gmm_rows["Test_MeanEntropy"], marker="o")
plt.xlabel("Dim")
plt.ylabel("Mean Entropy")
plt.title("GMM Entropy vs Dim")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/c1_gmm_entropy.png", dpi=200, bbox_inches="tight")
plt.show()

# covariance collapse
plt.figure(figsize=(7, 5))
plt.plot(gmm_rows["Dim"], gmm_rows["CovFloorRatio"], marker="o")
plt.xlabel("Dim")
plt.ylabel("CovFloorRatio")
plt.title("GMM Covariance Collapse")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/c1_gmm_covfloor.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Raw + KMeans
visualize_test_clusters_on_pca2(artifacts, "Raw-64", "KMeans")
plt.savefig(f"{FIG_DIR}/c1_raw_kmeans.png", dpi=200, bbox_inches="tight")

# Raw + GMM
visualize_test_clusters_on_pca2(artifacts, "Raw-64", "GMM")
plt.savefig(f"{FIG_DIR}/c1_raw_gmm.png", dpi=200, bbox_inches="tight")

# PCA-10 + KMeans
visualize_test_clusters_on_pca2(artifacts, "PCA-10", "KMeans")
plt.savefig(f"{FIG_DIR}/c1_pca10_kmeans.png", dpi=200, bbox_inches="tight")

# PCA-10 + GMM
visualize_test_clusters_on_pca2(artifacts, "PCA-10", "GMM")
plt.savefig(f"{FIG_DIR}/c1_pca10_gmm.png", dpi=200, bbox_inches="tight")